# Allocator — Align to Strategy Percentages (`pct_port_Sx`)

Reads the latest row-date from your **strat_symbol_day_overview_simple_values.csv**, selects a strategy column (e.g., `pct_port_S5`), converts those percentages into **target dollar weights** based on deployable equity, and rebalances your Alpaca **Paper** portfolio. Supports **fractional shares** via notional orders when allowed.

**Workflow**
1. Put your **paper keys** in Cell 0 (or export env vars).
2. Set the **CSV path** and **STRATEGY** in Cell 1–2.
3. Run Cells 0→6 once to preview plan; execute in Cell 7; start hourly loop in Cell 8; stop in Cell 9.

Orders use `client_order_id` with the prefix `alloc-` to avoid clashing with your guardian’s `guard-` prefix.

In [69]:
# === Cell 0 — PAPER KEYS (set here for paper; move to env for live) ===
APCA_KEY_ID     = "PKUNRTQLNIJ4FWITDRXUCGZTPT"  # <-- your PAPER Key ID (or leave blank to use env)
APCA_SECRET_KEY = "6fGdcfwCzp8vHUYXacaoVLpPsknY5n4d9ngukbqFdYpU"  # <-- your PAPER Secret Key (or leave blank to use env)


In [70]:
# === Cell 1 — Alpaca config, timezone, CSV path ===
import os, requests, pandas as pd, numpy as np, asyncio, uuid
from pathlib import Path
from datetime import datetime, timedelta, time as dtime
from dateutil import tz

TRADING_BASE = 'https://paper-api.alpaca.markets'
DATA_BASE    = 'https://data.alpaca.markets'

# Prefer in-notebook keys; else env
APCA_HEADERS = {
    "APCA-API-KEY-ID":    APCA_KEY_ID,
    "APCA-API-SECRET-KEY": APCA_SECRET_KEY,
}

assert APCA_HEADERS['APCA-API-KEY-ID'] and APCA_HEADERS['APCA-API-SECRET-KEY'], (
    'Set APCA_KEY_ID/APCA_SECRET_KEY in Cell 0 or export APCA_API_KEY_ID/APCA_API_SECRET_KEY env vars.'
)

TZ = tz.gettz('America/Los_Angeles')
MARKET_START_PT = dtime(6, 35)  # 9:35 ET
MARKET_END_PT   = dtime(13, 0)  # 16:00 ET

# User-provided CSV path
STRAT_CSV = Path(r"C:\Users\brobi\OneDrive\Desktop\Algo1\logs\out\alloc_percent_select_with_today.csv")


In [71]:
# === Cell 2 — Strategy selection & allocation knobs ===
# Choose which pct_port_Sx column to use, e.g., 'S3','S5','S6','S7','S8'
STRATEGY = 'S12'  # <-- change here

ALLOC = {
    'use_broker_equity': True,   # use Alpaca /v2/account equity
    'assumed_equity': 30000.0,   # fallback if broker call fails
    'cash_buffer': 2000.0,       # dollars to hold out of the market
    'max_symbols': 60,           # cap names to hold (after sorting by weight)
    'min_order_notional': 25.0,  # skip dust trades
    'rebalance_threshold': 0.03, # rebalance if |delta| > 3% of target value
    'rth_only': True,            # act only during regular trading hours
}


In [72]:
# === Cell 3 — Broker/account helpers ===
def broker_account():
    r = requests.get(f"{TRADING_BASE}/v2/account", headers=APCA_HEADERS, timeout=10)
    r.raise_for_status()
    return r.json()

def broker_positions_df():
    r = requests.get(f"{TRADING_BASE}/v2/positions", headers=APCA_HEADERS, timeout=10)
    if r.status_code == 404:
        return pd.DataFrame(columns=['symbol','qty','avg_entry_price'])
    r.raise_for_status()
    js = r.json()
    if isinstance(js, dict):
        js = js.get('positions', [])
    if not js:
        return pd.DataFrame(columns=['symbol','qty','avg_entry_price'])
    df = pd.DataFrame(js)
    df['symbol'] = df['symbol'].astype(str).str.upper()
    df['qty'] = df['qty'].astype(float)
    df['avg_entry_price'] = df['avg_entry_price'].astype(float)
    return df[['symbol','qty','avg_entry_price']]


def last_price(symbol: str) -> float | None:
    """
    Fetch the latest 1-minute bar close from Alpaca for `symbol`.
    Returns a float close price, or None on error (with debug prints).
    """
    url = f"{DATA_BASE}/v2/stocks/{symbol}/bars/latest"
    params = {"feed": "iex"}  # or "sip" if that's your plan

    try:
        print(f"[last_price] GET {url} params={params} symbol={symbol}")
        resp = requests.get(url, headers=APCA_HEADERS, params=params, timeout=10)
        print(f"[last_price] {symbol} status={resp.status_code}")

        # If not OK, show a bit of the body and bail
        if not resp.ok:
            print(f"[last_price] {symbol} error body: {resp.text[:300]}")
            resp.raise_for_status()   # will drop into except
            return None

        data = resp.json()

        # Single-symbol latest bar endpoint returns a 'bar' object
        bar = data.get("bar")
        if not bar:
            print(f"[last_price] {symbol} no 'bar' field in response: {json.dumps(data)[:300]}")
            return None

        c = bar.get("c")
        if c is None:
            print(f"[last_price] {symbol} 'bar' has no close: {json.dumps(bar)[:200]}")
            return None

        price = float(c)
        print(f"[last_price] {symbol} latest close={price}")
        return price

    except Exception as e:
        # Make sure we see *why* this failed
        try:
            body = resp.text[:300]
        except Exception:
            body = "<no body>"
        print(f"[last_price] ERROR for {symbol}: {e} | body={body}")
        return None

def asset_info(symbol: str) -> dict | None:
    try:
        rr = requests.get(f"{TRADING_BASE}/v2/assets/{symbol}", headers=APCA_HEADERS, timeout=10)
        if rr.status_code != 200:
            return None
        return rr.json()
    except Exception:
        return None


In [73]:
# === Cell 4 — Build targets from pct_port / pct_Sx strategy ===
from pathlib import Path
import pandas as pd

def deployable_equity(acct_json: dict, alloc: dict = ALLOC) -> float:
    if alloc["use_broker_equity"]:
        try:
            eq = float(acct_json.get("equity", alloc["assumed_equity"]))
        except Exception:
            eq = alloc["assumed_equity"]
    else:
        eq = alloc["assumed_equity"]
    return max(0.0, eq - float(alloc["cash_buffer"]))


def build_targets_from_pct(csv_path: Path, strategy: str, acct_json: dict) -> pd.DataFrame:
    """
    Read a CSV with day-by-day per-symbol weights and build target $ values
    for a given strategy.

    Accepts either:
      - pct_port_S8  (old RL overview style), or
      - pct_S8       (new infer-alloc style), or
      - a generic column exactly named `strategy` if present.
    """
    assert csv_path.exists(), f"Missing CSV: {csv_path}"
    df = pd.read_csv(csv_path)

    # Parse date and keep most recent
    if "date" not in df.columns:
        raise ValueError('CSV missing "date" column')
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    latest_date = df["date"].max()
    if pd.isna(latest_date):
        raise ValueError("No valid dates found in CSV")

    day = df[df["date"].eq(latest_date)].copy()

    # Figure out which column holds the weights for this strategy
    # Try several common patterns in order of preference
    candidate_cols = [
        f"pct_port_{strategy}",  # original RL overview
        f"pct_{strategy}",       # alloc_percent_*_with_today.csv
        strategy,                # just "S8" etc if you ever name it that way
    ]
    col = None
    for c in candidate_cols:
        if c in day.columns:
            col = c
            break

    if col is None:
        raise ValueError(
            f"No weight column found for strategy {strategy}. "
            f"Tried: {', '.join(candidate_cols)}. "
            f"Available columns: {list(day.columns)}"
        )

    # Normalize symbol, clean weights
    if "symbol" not in day.columns:
        raise ValueError('CSV missing "symbol" column')

    day["symbol"] = day["symbol"].astype(str).str.upper().str.strip()
    day[col] = pd.to_numeric(day[col], errors="coerce").fillna(0.0)

    # Keep rows with positive weight
    day = day[day[col] > 0].copy()
    if day.empty:
        print(f"[targets] No positive weights in {col} for {latest_date.date()}")
        return pd.DataFrame(columns=["symbol", "target_value", "weight", "latest_date"])

    # These may be either 0–1 or 0–100; either way we re-normalize.
    weights = day[col].clip(lower=0.0)
    wsum = weights.sum()
    if wsum <= 0:
        print(f"[targets] Sum of weights is nonpositive for {col} on {latest_date.date()}")
        return pd.DataFrame(columns=["symbol", "target_value", "weight", "latest_date"])

    dep = deployable_equity(acct_json)
    day["weight"] = weights / wsum
    day["target_value"] = day["weight"] * dep

    # Cap number of symbols if desired
    day = day.sort_values("target_value", ascending=False).head(ALLOC["max_symbols"])

    out = day[["symbol", "target_value", "weight"]].copy()
    out["latest_date"] = latest_date.date().isoformat()
    print(f"[targets] {len(out)} symbols | deployable=${dep:,.2f} | latest={latest_date.date()}")
    return out.reset_index(drop=True)


# Example use:
# If you're using the new alloc file from the infer notebook:
#   STRAT_CSV = Path("./out/alloc_percent_all_with_today.csv")
#   STRATEGY  = "S8"
acct = broker_account()
TARGETS_USD = build_targets_from_pct(STRAT_CSV, STRATEGY, acct)
TARGETS_USD.head(15)


[targets] 28 symbols | deployable=$47,838.50 | latest=2025-11-21


,symbol,target_value,weight,latest_date
0,ALB,1913.54,0.04,2025-11-21
1,ARQQ,1913.54,0.04,2025-11-21
2,ARRY,1913.54,0.04,2025-11-21
3,ENVX,1913.54,0.04,2025-11-21
4,FLNC,1913.54,0.04,2025-11-21
5,FSLR,1913.54,0.04,2025-11-21
6,MDB,1913.54,0.04,2025-11-21
7,MRK,1913.54,0.04,2025-11-21
8,PGR,1913.54,0.04,2025-11-21
9,POWI,1913.54,0.04,2025-11-21


In [74]:
acct = broker_account()
print("Broker equity from Alpaca:", acct.get("equity"))

from pprint import pprint
print("\nFull account json snippet:")
pprint({k: acct.get(k) for k in ["equity", "cash", "buying_power", "portfolio_value"]})

dep = deployable_equity(acct, ALLOC)
print("\nDeployable equity (after cash buffer):", dep)

print("\nSum of target_value:", TARGETS_USD["target_value"].sum())
print("Number of symbols:", len(TARGETS_USD))
print("\nFirst few targets:")
display(TARGETS_USD[["symbol", "weight", "target_value"]].head(20))

Broker equity from Alpaca: 49838.5

Full account json snippet:
{'buying_power': '157837.99',
 'cash': '23209.27',
 'equity': '49838.5',
 'portfolio_value': '49838.5'}

Deployable equity (after cash buffer): 47838.5

Sum of target_value: 47838.49999999998
Number of symbols: 28

First few targets:


,symbol,weight,target_value
0,ALB,0.04,1913.540
1,ARQQ,0.04,1913.540
2,ARRY,0.04,1913.540
3,ENVX,0.04,1913.540
4,FLNC,0.04,1913.540
5,FSLR,0.04,1913.540
6,MDB,0.04,1913.540
7,MRK,0.04,1913.540
8,PGR,0.04,1913.540
9,POWI,0.04,1913.540


In [75]:
# === Cell 5 — Diff current vs target and plan orders (include HOLDs) ===
def current_values(prices: dict[str, float], pos_df: pd.DataFrame) -> dict[str, float]:
    vals = {}
    for _, r in pos_df.iterrows():
        sym = str(r['symbol']).upper()
        px = prices.get(sym)
        if px is not None:
            vals[sym] = float(r['qty']) * float(px)
    return vals

def plan_rebalance_pct(targets_usd: pd.DataFrame, pos_df: pd.DataFrame) -> pd.DataFrame:
    """
    - Any symbol with a positive target in TARGETS_USD gets equity (BUY/hold included).
    - If currently unheld (cur==0) and target>0, place the entry even if delta < min_order_notional.
    - If target==0 but currently held, plan a SELL to flatten (subject to min_order_notional).
    - Otherwise use min_order_notional + rebalance_threshold to avoid tiny churn.
    """
    if targets_usd is None or targets_usd.empty:
        print('[plan] No targets.')
        return pd.DataFrame()

    target_syms = set(targets_usd['symbol'])
    held_syms   = set(pos_df['symbol']) if not pos_df.empty else set()
    syms = sorted(target_syms.union(held_syms))  # include held names even if target is 0 (to flatten)

    # fetch prices once
    prices = {s: last_price(s) for s in syms}

    cur_vals = current_values(prices, pos_df)
    tmap = dict(zip(targets_usd['symbol'], targets_usd['target_value']))

    plans = []
    for s in syms:
        tgt = float(tmap.get(s, 0.0))
        cur = float(cur_vals.get(s, 0.0))
        px  = prices.get(s)

        # skip if we don't have a price
        if px is None or px <= 0:
            continue

        delta = tgt - cur

        # --- Inclusion rules ---
        # 1) First-time entry (cur==0 & tgt>0): always place the buy (ensures HOLD/BUY with pct gets equity)
        if cur <= 0 and tgt > 0:
            plan = {
                'symbol': s, 'side': 'buy', 'delta_value': float(tgt),  # buy full target
                'target_value': float(tgt), 'current_value': 0.0,
                'price': px, 'fractionable': bool((asset_info(s) or {}).get('fractionable', False))
            }
            if plan['fractionable']:
                plan['notional'] = round(plan['delta_value'], 2)
                plan['qty'] = None
            else:
                plan['qty'] = int(max(1, plan['delta_value'] // px))  # at least 1 share for non-fractionable
                plan['notional'] = None
            plans.append(plan)
            continue

        # 2) Normal rebalance path
        #    a) enforce min notional to avoid dust
        if abs(delta) < ALLOC['min_order_notional']:
            continue

        #    b) relative threshold only when tgt>0
        if tgt > 0:
            rel = abs(delta) / tgt
            if rel < ALLOC['rebalance_threshold']:
                continue

        #    c) build plan
        info = asset_info(s) or {}
        fractionable = bool(info.get('fractionable', False))
        plan = {
            'symbol': s,
            'side': 'buy' if delta > 0 else 'sell',
            'delta_value': float(delta),
            'target_value': float(tgt),
            'current_value': float(cur),
            'price': px,
            'fractionable': fractionable,
        }
        if fractionable:
            plan['notional'] = round(abs(delta), 2)
            plan['qty'] = None
        else:
            plan['qty'] = int(abs(delta) // px)
            plan['notional'] = None
        plans.append(plan)

    plans_df = pd.DataFrame(plans)
    if plans_df.empty:
        print('[plan] No orders needed.')
        return plans_df

    # Sells first, then buys (cash-friendly)
    return plans_df.sort_values(['side','delta_value'], ascending=[True, False])

POS = broker_positions_df()
PLANS = plan_rebalance_pct(TARGETS_USD, POS)
PLANS.head(20)


[last_price] GET https://data.alpaca.markets/v2/stocks/ALB/bars/latest params={'feed': 'iex'} symbol=ALB
[last_price] ALB status=200
[last_price] ALB latest close=116.77
[last_price] GET https://data.alpaca.markets/v2/stocks/ARQQ/bars/latest params={'feed': 'iex'} symbol=ARQQ
[last_price] ARQQ status=200
[last_price] ARQQ latest close=24.32
[last_price] GET https://data.alpaca.markets/v2/stocks/ARRY/bars/latest params={'feed': 'iex'} symbol=ARRY
[last_price] ARRY status=200
[last_price] ARRY latest close=7.135
[last_price] GET https://data.alpaca.markets/v2/stocks/ENVX/bars/latest params={'feed': 'iex'} symbol=ENVX
[last_price] ENVX status=200
[last_price] ENVX latest close=7.48
[last_price] GET https://data.alpaca.markets/v2/stocks/FLNC/bars/latest params={'feed': 'iex'} symbol=FLNC
[last_price] FLNC status=200
[last_price] FLNC latest close=15.41
[last_price] GET https://data.alpaca.markets/v2/stocks/FSLR/bars/latest params={'feed': 'iex'} symbol=FSLR
[last_price] FSLR status=200
[la

,symbol,side,delta_value,target_value,current_value,price,fractionable,notional,qty
0,ALB,buy,1913.540,1913.540,0.0,116.770,True,1913.54,NaN
1,ARQQ,buy,1913.540,1913.540,0.0,24.320,True,1913.54,NaN
2,ARRY,buy,1913.540,1913.540,0.0,7.135,True,1913.54,NaN
3,ENVX,buy,1913.540,1913.540,0.0,7.480,True,1913.54,NaN
4,FLNC,buy,1913.540,1913.540,0.0,15.410,True,1913.54,NaN
5,FSLR,buy,1913.540,1913.540,0.0,249.765,True,1913.54,NaN
10,MDB,buy,1913.540,1913.540,0.0,321.150,True,1913.54,NaN
11,MRK,buy,1913.540,1913.540,0.0,97.730,True,1913.54,NaN
13,NVDA,buy,1913.540,1913.540,0.0,180.230,True,1913.54,NaN
16,PGR,buy,1913.540,1913.540,0.0,226.920,True,1913.54,NaN


In [76]:
# === Debug: why are there "no orders"? ===
import pandas as pd

acct = broker_account()
TARGETS_USD = build_targets_from_pct(STRAT_CSV, STRATEGY, acct)
POS = broker_positions_df()

target_map  = dict(zip(TARGETS_USD["symbol"], TARGETS_USD["target_value"]))
target_syms = set(target_map)
held_syms   = set(POS["symbol"]) if not POS.empty else set()
syms        = sorted(target_syms | held_syms)

prices   = {s: last_price(s) for s in syms}
cur_vals = current_values(prices, POS)

rows = []
for s in syms:
    tgt = float(target_map.get(s, 0.0))
    cur = float(cur_vals.get(s, 0.0))
    px  = prices.get(s)
    
    if px is None or px <= 0:
        reason = "SKIP: no price from last_price()"
        delta = tgt - cur
    else:
        delta = tgt - cur
        if cur <= 0 and tgt > 0:
            reason = "NEW ENTRY: will BUY full target in plan()"
        elif abs(delta) < ALLOC["min_order_notional"]:
            reason = f"SKIP: |delta|<{ALLOC['min_order_notional']}"
        elif tgt > 0 and abs(delta)/max(tgt, 1e-9) < ALLOC["rebalance_threshold"]:
            reason = f"SKIP: rel<{ALLOC['rebalance_threshold']}"
        else:
            reason = "WILL TRADE in plan_rebalance_pct"

    rows.append({
        "symbol": s,
        "target_value": round(tgt, 2),
        "current_value": round(cur, 2),
        "delta": round(delta, 2),
        "price_used": px,
        "reason": reason,
    })

debug_df = pd.DataFrame(rows).sort_values("symbol")
debug_df



[targets] 28 symbols | deployable=$47,838.50 | latest=2025-11-21
[last_price] GET https://data.alpaca.markets/v2/stocks/ALB/bars/latest params={'feed': 'iex'} symbol=ALB
[last_price] ALB status=200
[last_price] ALB latest close=116.77
[last_price] GET https://data.alpaca.markets/v2/stocks/ARQQ/bars/latest params={'feed': 'iex'} symbol=ARQQ
[last_price] ARQQ status=200
[last_price] ARQQ latest close=24.32
[last_price] GET https://data.alpaca.markets/v2/stocks/ARRY/bars/latest params={'feed': 'iex'} symbol=ARRY
[last_price] ARRY status=200
[last_price] ARRY latest close=7.135
[last_price] GET https://data.alpaca.markets/v2/stocks/ENVX/bars/latest params={'feed': 'iex'} symbol=ENVX
[last_price] ENVX status=200
[last_price] ENVX latest close=7.48
[last_price] GET https://data.alpaca.markets/v2/stocks/FLNC/bars/latest params={'feed': 'iex'} symbol=FLNC
[last_price] FLNC status=200
[last_price] FLNC latest close=15.41
[last_price] GET https://data.alpaca.markets/v2/stocks/FSLR/bars/latest pa

,symbol,target_value,current_value,delta,price_used,reason
0,ALB,1913.54,0.00,1913.54,116.770,NEW ENTRY: will BUY full target in plan()
1,ARQQ,1913.54,0.00,1913.54,24.320,NEW ENTRY: will BUY full target in plan()
2,ARRY,1913.54,0.00,1913.54,7.135,NEW ENTRY: will BUY full target in plan()
3,ENVX,1913.54,0.00,1913.54,7.480,NEW ENTRY: will BUY full target in plan()
4,FLNC,1913.54,0.00,1913.54,15.410,NEW ENTRY: will BUY full target in plan()
5,FSLR,1913.54,0.00,1913.54,249.765,NEW ENTRY: will BUY full target in plan()
6,GEVO,1435.15,0.00,1435.15,1.950,NEW ENTRY: will BUY full target in plan()
7,GOOGL,956.77,0.00,956.77,299.860,NEW ENTRY: will BUY full target in plan()
8,INTC,1435.15,0.00,1435.15,34.500,NEW ENTRY: will BUY full target in plan()
9,ITRI,1435.15,0.00,1435.15,95.600,NEW ENTRY: will BUY full target in plan()


In [77]:
# === Cell 6 — Preview summary ===
if PLANS is not None and not PLANS.empty:
    summary = PLANS.groupby('side')['delta_value'].sum().to_dict()
    print('[summary] to BUY $%.2f, to SELL $%.2f' % (summary.get('buy', 0.0), abs(summary.get('sell', 0.0))))
else:
    print('[summary] no changes')


[summary] to BUY $44011.42, to SELL $22777.85


In [78]:
# === Cell 7 — Execute rebalancing orders (with small safety margin on sells) ===
import uuid
import requests
import pandas as pd

def submit_order(payload: dict):
    p = dict(payload)
    p.setdefault("client_order_id", f"alloc-{uuid.uuid4().hex[:12]}")
    r = requests.post(f"{TRADING_BASE}/v2/orders", headers=APCA_HEADERS, json=p, timeout=10)
    if r.status_code >= 400:
        print("[order error]", r.status_code, r.text)
    r.raise_for_status()
    return r.json()

def execute_plans(plans_df: pd.DataFrame):
    if plans_df is None or plans_df.empty:
        print("[exec] Nothing to do.")
        return []

    results = []
    SAFETY = 0.999  # trim sells by 0.1% so we never slightly overshoot

    # Prioritize sells then buys to free cash
    for side in ["sell", "buy"]:
        pp = plans_df[plans_df["side"].eq(side)].copy()

        for _, r in pp.iterrows():
            sym = r["symbol"]

            # --- fractional (use notional) ---
            if bool(r.get("fractionable")) and pd.notna(r.get("notional")) and float(r["notional"]) > 0:
                notional = float(r["notional"])

                # For SELLs, back off a tiny bit to avoid "insufficient qty" due to rounding
                if side == "sell":
                    notional = notional * SAFETY

                if notional <= 0:
                    continue

                payload = {
                    "symbol": sym,
                    "side": side,
                    "type": "market",
                    "time_in_force": "day",
                    "notional": round(notional, 2),
                }

            # --- share-based (qty) ---
            else:
                qty = int(r.get("qty") or 0)

                # For safety, don't try to send zero or negative qty
                if qty <= 0:
                    continue

                # Optional: trim sell qty by 1 share if you often get errors near the edge
                if side == "sell" and qty > 0:
                    # If you want to be ultra-safe:
                    # qty = max(qty - 1, 0)
                    pass

                if qty <= 0:
                    continue

                payload = {
                    "symbol": sym,
                    "side": side,
                    "type": "market",
                    "time_in_force": "day",
                    "qty": str(qty),
                }

            try:
                res = submit_order(payload)
                print(f"[exec] {side.upper()} {sym} -> ok")
                results.append(res)
            except Exception as e:
                print(f"[exec error] {sym}: {e}")

    return results

EXEC_RESULTS = execute_plans(PLANS)


[exec] SELL QS -> ok
[exec] SELL TSM -> ok
[exec] BUY ALB -> ok
[exec] BUY ARQQ -> ok
[exec] BUY ARRY -> ok
[exec] BUY ENVX -> ok
[exec] BUY FLNC -> ok
[exec] BUY FSLR -> ok
[exec] BUY MDB -> ok
[exec] BUY MRK -> ok
[exec] BUY NVDA -> ok
[exec] BUY PGR -> ok
[exec] BUY POWI -> ok
[exec] BUY RMBS -> ok
[exec] BUY RNW -> ok
[exec] BUY SLDP -> ok
[exec] BUY SMCI -> ok
[exec] BUY TM -> ok
[exec] BUY VICR -> ok
[exec] BUY GEVO -> ok
[exec] BUY INTC -> ok
[exec] BUY ITRI -> ok
[exec] BUY MVST -> ok
[exec] BUY NXT -> ok
[exec] BUY PSTG -> ok
[exec] BUY GOOGL -> ok
[exec] BUY PG -> ok
[exec] BUY XEL -> ok


In [81]:
print("Number of planned trades:", len(PLANS))

print("\nBy side:")
print(PLANS["side"].value_counts())

print("\nRows where we would have traded but qty/notional is missing:")
PLANS[(PLANS["qty"].isna()) & (PLANS["notional"].isna())]

Number of planned trades: 28

By side:
side
buy     26
sell     2
Name: count, dtype: int64

Rows where we would have traded but qty/notional is missing:


,symbol,side,delta_value,target_value,current_value,price,fractionable,notional,qty


In [79]:
# === Cell 8 — Start hourly allocator loop (no RTH restriction) ===
import asyncio
from datetime import datetime, timedelta

async def allocator_once():
    now = datetime.now(TZ)
    print(f"[allocator] running at {now.strftime('%Y-%m-%d %H:%M:%S')} (no RTH restriction)")

    acct = broker_account()
    targets_usd = build_targets_from_pct(STRAT_CSV, STRATEGY, acct)

    if targets_usd is None or targets_usd.empty:
        print("[allocator] no targets (empty targets_usd); skipping rebalance")
        return

    pos = broker_positions_df()
    plans = plan_rebalance_pct(targets_usd, pos)

    if plans is None or plans.empty:
        print("[allocator] no rebalance needed (PLANS empty).")
        return

    execute_plans(plans)


async def _sleep_to_top_of_hour():
    now = datetime.now(TZ)
    next_hour = now.replace(minute=0, second=0, microsecond=0) + timedelta(hours=1)
    delay = max(5.0, (next_hour - now).total_seconds())
    print(f"[allocator] next run at {next_hour.strftime('%H:%M:%S')} (~{int(delay)}s)")
    await asyncio.sleep(delay)


async def allocator_loop():
    print("✅ Hourly allocator started (runs regardless of market hours).")
    while True:
        try:
            await allocator_once()
        except Exception as e:
            print("[allocator_loop] error:", e)
        await _sleep_to_top_of_hour()


# Start it
alloc_task = asyncio.create_task(allocator_loop())
print("Allocator task created — leave the kernel running.")


Allocator task created — leave the kernel running.


✅ Hourly allocator started (runs regardless of market hours).
[allocator] running at 2025-11-23 11:11:50 (no RTH restriction)
[targets] 28 symbols | deployable=$47,838.50 | latest=2025-11-21
[last_price] GET https://data.alpaca.markets/v2/stocks/ALB/bars/latest params={'feed': 'iex'} symbol=ALB
[last_price] ALB status=200
[last_price] ALB latest close=116.77
[last_price] GET https://data.alpaca.markets/v2/stocks/ARQQ/bars/latest params={'feed': 'iex'} symbol=ARQQ
[last_price] ARQQ status=200
[last_price] ARQQ latest close=24.32
[last_price] GET https://data.alpaca.markets/v2/stocks/ARRY/bars/latest params={'feed': 'iex'} symbol=ARRY
[last_price] ARRY status=200
[last_price] ARRY latest close=7.135
[last_price] GET https://data.alpaca.markets/v2/stocks/ENVX/bars/latest params={'feed': 'iex'} symbol=ENVX
[last_price] ENVX status=200
[last_price] ENVX latest close=7.48
[last_price] GET https://data.alpaca.markets/v2/stocks/FLNC/bars/latest params={'feed': 'iex'} symbol=FLNC
[last_price] F

In [80]:
# === Cell 9 — Stop hourly allocator ===
alloc_task.cancel()
await asyncio.gather(alloc_task, return_exceptions=True)
print('Allocator stopped ✅')


Allocator stopped ✅
